# GLoW cocoBrain — Inference & Visualisation

Notebook pour explorer les resultats du Global Workspace (vision + texte E5).

**Architecture Run 13+** : ResNet-50 features (2048) + E5-large-v2 embeddings (1024) → latent 128 → workspace 64.
Fusion = weighted sum + Tanh. Pas de tokenizer — on utilise le nearest-neighbor en espace E5 pour "decoder" les traductions.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from PIL import Image
import os
import json
import yaml

from models.encoders import VisionEncoder, TextEncoder
from models.decoders import VisionDecoder, TextDecoder
from models.domain_module import DomainModule
from models.workspace import GlobalWorkspace
from data.dataset import COCODataset

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

## 1. Charger le modele et les donnees

In [ ]:
# === CHOISIR LA RUN A VISUALISER ===
# Mettre le chemin du dossier run ici, ou None pour le dernier
RUN_DIR = None  # ex: "../checkpoints/run_011_run13_refonte"

if RUN_DIR is None:
    import glob
    runs = sorted(glob.glob("../checkpoints/run_*"))
    if runs:
        RUN_DIR = runs[-1]
        print(f"Derniere run detectee: {RUN_DIR}")
    else:
        raise FileNotFoundError("Pas de dossier run trouve")

# Config (celle de la run, pas la config actuelle)
config_path = f"{RUN_DIR}/config.yaml" if os.path.exists(f"{RUN_DIR}/config.yaml") else "../config/config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)
print(f"Config: {config_path}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Dataset val
DATA_ROOT = f"../{config['data']['data_root']}"
val_dataset = COCODataset(
    data_root=DATA_ROOT,
    split='val',
    paired_fraction=config['training'].get('paired_fraction', 1.0),
)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

# Charger les captions pour l'affichage (nearest-neighbor decoding)
with open(f"{DATA_ROOT}/caption_ids_val.json") as f:
    all_captions_meta = json.load(f)
all_caption_texts = [c["caption"] for c in all_captions_meta]
print(f"Captions val chargees: {len(all_caption_texts)}")

In [ ]:
# Reconstruire le modele et charger les poids
cfg_proj = config['projection']
ws_dim = config['workspace_dim']
cfg_v = config['vision']
cfg_t = config['text']

domain_modules = {
    "vision": DomainModule(
        name="vision",
        encoder=VisionEncoder(cfg_v['feature_dim'], config['latent_dims']['vision']),
        decoder=VisionDecoder(config['latent_dims']['vision'], cfg_v['feature_dim']),
        latent_dim=config['latent_dims']['vision'],
        workspace_dim=ws_dim,
        proj_hidden_dim=cfg_proj['hidden_dim'],
        proj_num_layers=cfg_proj['num_layers'],
        proj_dropout=cfg_proj['dropout'],
    ),
    "text": DomainModule(
        name="text",
        encoder=TextEncoder(cfg_t['feature_dim'], config['latent_dims']['text']),
        decoder=TextDecoder(config['latent_dims']['text'], cfg_t['feature_dim']),
        latent_dim=config['latent_dims']['text'],
        workspace_dim=ws_dim,
        proj_hidden_dim=cfg_proj['hidden_dim'],
        proj_num_layers=cfg_proj['num_layers'],
        proj_dropout=cfg_proj['dropout'],
    ),
}

workspace = GlobalWorkspace(
    domain_modules=domain_modules,
    workspace_dim=ws_dim,
).to(device)

# Charger checkpoint
ckpt_path = f"{RUN_DIR}/checkpoint.pt"
ckpt = torch.load(ckpt_path, map_location=device)
workspace.load_state_dict(ckpt['model_state_dict'])
workspace.eval()
print(f"Checkpoint charge: {ckpt_path}")
print(f"Params: {sum(p.numel() for p in workspace.parameters()):,}")

# Afficher les metriques de la run si dispo
if 'eval_results' in ckpt:
    print("\nResultats de la run:")
    for k, v in ckpt['eval_results'].items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Helpers
with open(f"{DATA_ROOT}/image_ids_val.json") as f:
    val_image_ids = json.load(f)

def load_coco_image(image_id, split='val'):
    """Charge l'image COCO originale pour l'affichage."""
    path = f"{DATA_ROOT}/{split}2017/{image_id:012d}.jpg"
    return Image.open(path).convert('RGB')

def get_sample_info(idx):
    """Recupere image_id et caption pour un sample du val dataset."""
    image_id, text_idx = val_dataset.samples[idx]
    caption = all_caption_texts[text_idx]
    return image_id, caption

# E5 embeddings normalises pour nearest-neighbor decoding
all_text_features_norm = F.normalize(
    torch.tensor(val_dataset.text_features, dtype=torch.float32), dim=-1
)

def decode_e5_to_caption(e5_embedding, top_k=1):
    """Trouve la/les caption(s) les plus proches en espace E5 (nearest-neighbor)."""
    emb_norm = F.normalize(e5_embedding.cpu().flatten().unsqueeze(0), dim=-1)
    sims = (emb_norm @ all_text_features_norm.T).squeeze(0)
    topk_vals, topk_ids = sims.topk(top_k)
    if top_k == 1:
        return all_caption_texts[topk_ids[0].item()]
    return [(all_caption_texts[idx.item()], val.item()) for val, idx in zip(topk_vals, topk_ids)]

print("Helpers prets.")

## 2. Traduction Image -> Texte

On traduit des images du val set via le workspace, puis on decode par nearest-neighbor en espace E5.
La traduction passe par: vision features → encoder → latent → projection → workspace → inv_projection → text latent → decoder → E5 embedding → NN lookup.

In [ ]:
@torch.no_grad()
def translate_and_show(indices, ncols=5):
    """Affiche des images avec leur caption GT et la traduction du workspace."""
    nrows = (len(indices) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 5*nrows))
    if nrows == 1:
        axes = [axes]
    axes = np.array(axes).flatten()
    
    for i, idx in enumerate(indices):
        image_id, gt_caption = get_sample_info(idx)
        
        # Forward: vision features -> workspace -> text E5 embedding -> NN caption
        sample = val_dataset[idx]
        vision_feat = sample['vision'].unsqueeze(0).to(device)
        text_e5_hat = workspace.translate('vision', 'text', vision_feat)
        pred_caption = decode_e5_to_caption(text_e5_hat)
        
        # MSE de la traduction
        text_gt = sample['text'].unsqueeze(0).to(device)
        mse = F.mse_loss(text_e5_hat, text_gt).item()
        
        # Afficher l'image
        img = load_coco_image(image_id)
        axes[i].imshow(img)
        axes[i].set_title(
            f"GT: {gt_caption[:60]}\nNN: {pred_caption[:60]}\nMSE: {mse:.4f}",
            fontsize=8, wrap=True
        )
        axes[i].axis('off')
    
    for j in range(len(indices), len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

# Afficher 15 exemples espaces dans le val set
indices = np.linspace(0, len(val_dataset)-1, 15, dtype=int)
translate_and_show(indices)

## 3. Retrieval: Image -> Texte

Pour chaque image, on cherche la caption la plus similaire dans le workspace (cosine similarity).

In [ ]:
@torch.no_grad()
def compute_all_workspace_reprs(dataloader):
    """Projette tout le val set dans le workspace."""
    all_w_vision, all_w_text = [], []
    
    for batch in dataloader:
        w_v = workspace.domain_modules['vision'].encode_to_workspace(batch['vision'].to(device))
        w_t = workspace.domain_modules['text'].encode_to_workspace(batch['text'].to(device))
        all_w_vision.append(F.normalize(w_v, dim=-1).cpu())
        all_w_text.append(F.normalize(w_t, dim=-1).cpu())
    
    return torch.cat(all_w_vision), torch.cat(all_w_text)

all_w_vision, all_w_text = compute_all_workspace_reprs(val_loader)
print(f"Vision workspace: {all_w_vision.shape}")
print(f"Text workspace:   {all_w_text.shape}")

# Sous-ensemble pour la matrice de similarite (evite OOM)
N_EVAL = 5000
eval_indices = np.random.choice(len(all_w_vision), N_EVAL, replace=False)
eval_indices.sort()

w_v_eval = all_w_vision[eval_indices]
w_t_eval = all_w_text[eval_indices]

sim_matrix = w_v_eval @ w_t_eval.T
print(f"\nSimilarity matrix ({N_EVAL}x{N_EVAL}):")
print(f"  mean: {sim_matrix.mean():.4f}")
print(f"  diagonal mean (positives): {sim_matrix.diagonal().mean():.4f}")
print(f"  off-diagonal mean (negatives): {(sim_matrix.sum() - sim_matrix.diagonal().sum()) / (sim_matrix.numel() - len(sim_matrix)):.4f}")

In [ ]:
# Recall@K sur le sous-ensemble
def recall_at_k(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    N = sim_matrix.size(0)
    gt = torch.arange(N)
    _, topk = sim_matrix.topk(max(k_values), dim=1)
    
    results = {}
    for k in k_values:
        correct = (topk[:, :k] == gt.unsqueeze(1)).any(dim=1)
        results[f'R@{k}'] = correct.float().mean().item()
    return results

# Vision -> Text
v2t = recall_at_k(sim_matrix)
print(f"=== Vision -> Text (sur {N_EVAL} samples) ===")
for k, v in v2t.items():
    print(f"  {k}: {v:.4f} ({v*100:.1f}%)")

# Text -> Vision
t2v = recall_at_k(sim_matrix.T)
print(f"\n=== Text -> Vision (sur {N_EVAL} samples) ===")
for k, v in t2v.items():
    print(f"  {k}: {v:.4f} ({v*100:.1f}%)")

## 4. Exemples de retrieval

Pour quelques images, on affiche les top-5 captions retrievees depuis le workspace.

In [ ]:
def show_retrieval_examples(query_indices_in_sim, top_k=5):
    """Pour chaque query (index dans sim_matrix), affiche l'image + les top-K captions."""
    for sim_idx in query_indices_in_sim:
        dataset_idx = eval_indices[sim_idx]
        image_id, gt_caption = get_sample_info(dataset_idx)
        
        sims = sim_matrix[sim_idx]
        topk_vals, topk_ids = sims.topk(top_k)
        
        # Rank du vrai match
        rank = (sims >= sims[sim_idx]).sum().item()
        
        fig, ax = plt.subplots(1, 1, figsize=(5, 5))
        img = load_coco_image(image_id)
        ax.imshow(img)
        ax.axis('off')
        
        title = f"GT: {gt_caption[:80]}\nRank du vrai match: {rank}/{N_EVAL}\n\n"
        for j, (score, tid) in enumerate(zip(topk_vals, topk_ids)):
            retrieved_dataset_idx = eval_indices[tid.item()]
            _, retrieved_caption = get_sample_info(retrieved_dataset_idx)
            match = " <<< MATCH" if tid.item() == sim_idx else ""
            title += f"  Top-{j+1} ({score:.3f}): {retrieved_caption[:70]}{match}\n"
        
        ax.set_title(title, fontsize=8, ha='left', x=0, wrap=True)
        plt.tight_layout()
        plt.show()
        print()

# 6 exemples espaces dans le sous-ensemble
show_retrieval_examples(np.linspace(0, N_EVAL-1, 6, dtype=int))

## 4b. Retrieval inverse: Texte -> Images

On donne un texte (caption) et on cherche les images les plus similaires dans le workspace.

In [ ]:
def show_text_to_image_retrieval(query_indices_in_sim, top_k=5):
    """Pour chaque texte query, affiche la GT image + les top-K images retrievees."""
    sim_t2v = sim_matrix.T
    
    for sim_idx in query_indices_in_sim:
        dataset_idx = eval_indices[sim_idx]
        image_id, gt_caption = get_sample_info(dataset_idx)
        
        sims = sim_t2v[sim_idx]
        topk_vals, topk_ids = sims.topk(top_k)
        
        rank = (sims >= sims[sim_idx]).sum().item()
        
        fig, axes = plt.subplots(1, top_k + 1, figsize=(3.5 * (top_k + 1), 4))
        
        gt_img = load_coco_image(image_id)
        axes[0].imshow(gt_img)
        axes[0].set_title(f"GT (rank {rank}/{N_EVAL})", fontsize=9, color='green', fontweight='bold')
        axes[0].axis('off')
        
        for j, (score, vid) in enumerate(zip(topk_vals, topk_ids)):
            retrieved_dataset_idx = eval_indices[vid.item()]
            ret_image_id, _ = get_sample_info(retrieved_dataset_idx)
            ret_img = load_coco_image(ret_image_id)
            axes[j + 1].imshow(ret_img)
            is_match = vid.item() == sim_idx
            color = 'green' if is_match else 'black'
            axes[j + 1].set_title(f"Top-{j+1} ({score:.3f})" + (" MATCH" if is_match else ""),
                                  fontsize=9, color=color)
            axes[j + 1].axis('off')
        
        plt.suptitle(f'Query: "{gt_caption[:80]}"', fontsize=10, y=1.02)
        plt.tight_layout()
        plt.show()
        print()

# 6 exemples
show_text_to_image_retrieval(np.linspace(0, N_EVAL-1, 6, dtype=int))

## 4c. Visualisation des losses en action

On prend un sample et on montre concretement ce que calcule chaque loss.

In [ ]:
@torch.no_grad()
def visualize_losses_for_sample(dataset_idx):
    """Visualise les losses pour un sample donne."""
    image_id, gt_caption = get_sample_info(dataset_idx)
    sample = val_dataset[dataset_idx]
    x_v = sample['vision'].unsqueeze(0).to(device)
    x_t = sample['text'].unsqueeze(0).to(device)
    
    mod_v = workspace.domain_modules['vision']
    mod_t = workspace.domain_modules['text']
    
    # Encode
    z_v = mod_v.encode(x_v)
    z_t = mod_t.encode(x_t)
    
    # Project to workspace
    w_v = mod_v.to_workspace(z_v)
    w_t = mod_t.to_workspace(z_t)
    
    # === 1. CONTRASTIVE ===
    w_v_norm = F.normalize(w_v, dim=-1)
    w_t_norm = F.normalize(w_t, dim=-1)
    cos_sim_positive = (w_v_norm * w_t_norm).sum().item()
    
    # === 2. TRANSLATION ===
    # Vision -> workspace -> Text E5
    z_t_from_v = mod_t.from_workspace(w_v)
    text_e5_hat = mod_t.decode(z_t_from_v)
    pred_caption = decode_e5_to_caption(text_e5_hat)
    translation_mse_v2t = F.mse_loss(text_e5_hat, x_t).item()
    
    # Text -> workspace -> Vision features
    z_v_from_t = mod_v.from_workspace(w_t)
    v_hat = mod_v.decode(z_v_from_t)
    translation_mse_t2v = F.mse_loss(v_hat, x_v).item()
    
    # === 3. CYCLE ===
    from losses.cycle import compute_cycle
    z_v_cycle = compute_cycle(mod_v, mod_t, z_v)
    cycle_mse = F.mse_loss(z_v_cycle, z_v).item()
    cycle_cos = F.cosine_similarity(z_v, z_v_cycle).item()
    
    # === VISUALISATION ===
    fig = plt.figure(figsize=(18, 12))
    
    # Image originale
    ax_img = fig.add_subplot(3, 3, 1)
    img = load_coco_image(image_id)
    ax_img.imshow(img)
    ax_img.set_title(f'Image originale\n"{gt_caption[:50]}..."', fontsize=9)
    ax_img.axis('off')
    
    # 1. CONTRASTIVE
    ax_contr = fig.add_subplot(3, 3, 2)
    n_compare = 20
    compare_indices = np.random.choice(len(val_dataset), n_compare, replace=False)
    other_sims = []
    for ci in compare_indices:
        other_sample = val_dataset[ci]
        other_t = other_sample['text'].unsqueeze(0).to(device)
        other_w_t = F.normalize(mod_t.encode_to_workspace(other_t), dim=-1)
        other_sims.append((w_v_norm * other_w_t).sum().item())
    
    values = other_sims + [cos_sim_positive]
    sorted_vals = sorted(enumerate(values), key=lambda x: x[1])
    bars = ax_contr.bar(range(len(values)), [v for _, v in sorted_vals], color='tab:red', alpha=0.6)
    for bar_idx, (orig_idx, val) in enumerate(sorted_vals):
        if orig_idx == n_compare:
            bars[bar_idx].set_color('tab:green')
            bars[bar_idx].set_alpha(1.0)
    ax_contr.set_title(f'CONTRASTIVE\nCosine sim\nMatch={cos_sim_positive:.3f} (vert)', fontsize=9)
    ax_contr.set_ylabel('Cosine sim')
    
    # 2. TRANSLATION
    ax_trans = fig.add_subplot(3, 3, 3)
    ax_trans.text(0.5, 0.7, 'TRANSLATION', fontsize=14, fontweight='bold',
                 ha='center', transform=ax_trans.transAxes)
    ax_trans.text(0.5, 0.5, f'Image -> Workspace -> Texte (NN):', fontsize=10,
                 ha='center', transform=ax_trans.transAxes)
    ax_trans.text(0.5, 0.35, f'NN: "{pred_caption[:60]}"', fontsize=9,
                 ha='center', transform=ax_trans.transAxes, style='italic')
    ax_trans.text(0.5, 0.2, f'GT: "{gt_caption[:60]}"', fontsize=9,
                 ha='center', transform=ax_trans.transAxes)
    ax_trans.text(0.5, 0.05, f'MSE v2t: {translation_mse_v2t:.4f} | MSE t2v: {translation_mse_t2v:.4f}',
                 fontsize=9, ha='center', transform=ax_trans.transAxes)
    ax_trans.axis('off')
    
    # 3. CYCLE
    ax_cycle1 = fig.add_subplot(3, 1, 2)
    w_v_step1 = mod_v.to_workspace(z_v)
    z_t_step2 = mod_t.from_workspace(w_v_step1)
    w_t_step3 = mod_t.to_workspace(z_t_step2)
    z_v_hat = mod_v.from_workspace(w_t_step3)
    
    steps = {
        'z_v (original)': z_v[0].cpu().numpy(),
        'w_v (proj)': w_v_step1[0].cpu().numpy(),
        'z_t\' (inv_proj)': z_t_step2[0].cpu().numpy(),
        'w_t\' (re-proj)': w_t_step3[0].cpu().numpy(),
        'z_v_hat (cycle)': z_v_hat[0].cpu().numpy(),
    }
    
    step_data = []
    for name, vec in steps.items():
        vec_norm = (vec - vec.mean()) / (vec.std() + 1e-8)
        step_data.append(vec_norm[:64])
    
    step_matrix = np.stack(step_data)
    im = ax_cycle1.imshow(step_matrix, aspect='auto', cmap='RdBu_r', vmin=-3, vmax=3)
    ax_cycle1.set_yticks(range(len(steps)))
    ax_cycle1.set_yticklabels(list(steps.keys()), fontsize=9)
    ax_cycle1.set_xlabel('Dimensions (64 premieres)')
    ax_cycle1.set_title(f'CYCLE: z_v -> w -> z_t\' -> w\' -> z_v_hat\n'
                       f'Cycle MSE: {cycle_mse:.4f} | Cycle cosine: {cycle_cos:.4f}', fontsize=10)
    plt.colorbar(im, ax=ax_cycle1, shrink=0.6)
    
    # Comparaison directe z_v vs z_v_hat
    ax_cycle2 = fig.add_subplot(3, 1, 3)
    dims_to_show = min(128, len(z_v[0]))
    ax_cycle2.plot(z_v[0, :dims_to_show].cpu().numpy(), label='z_v (original)', alpha=0.8)
    ax_cycle2.plot(z_v_hat[0, :dims_to_show].cpu().numpy(), label='z_v_hat (apres cycle)', alpha=0.8, linestyle='--')
    ax_cycle2.set_xlabel('Dimension')
    ax_cycle2.set_ylabel('Valeur')
    ax_cycle2.set_title(f'Cycle consistency: z_v vs z_v_hat (dims 0-{dims_to_show})', fontsize=10)
    ax_cycle2.legend()
    
    plt.tight_layout()
    plt.show()

# Visualiser pour 3 samples differents
for idx in [0, len(val_dataset)//3, 2*len(val_dataset)//3]:
    print(f"\n{'='*60}")
    visualize_losses_for_sample(idx)

## 4d. Telephone arabe: passages multiples dans le workspace

On fait passer un signal plusieurs fois dans le workspace pour voir la degradation.
Avec des embeddings E5, on decode par nearest-neighbor a chaque etape.

In [ ]:
# Pre-charger toutes les features val pour le nearest-neighbor vision
all_val_features = torch.tensor(val_dataset.features, dtype=torch.float32)
all_val_features_norm = F.normalize(all_val_features, dim=-1)

def find_nearest_image(feat_vector):
    """Trouve l'image du val set dont les features ResNet sont les plus proches."""
    feat_norm = F.normalize(feat_vector.cpu().flatten().unsqueeze(0), dim=-1)
    sims = (feat_norm @ all_val_features_norm.T).squeeze(0)
    best_idx = sims.argmax().item()
    image_id = val_image_ids[best_idx]
    return image_id, sims[best_idx].item()

@torch.no_grad()
def telephone_arabe(dataset_idx, n_passes=5):
    """Telephone arabe: ping-pong image <-> texte via le workspace."""
    sample = val_dataset[dataset_idx]
    image_id, gt_caption = get_sample_info(dataset_idx)
    
    mod_v = workspace.domain_modules['vision']
    mod_t = workspace.domain_modules['text']
    
    vision_feat = sample['vision'].unsqueeze(0).to(device)
    
    # MODE 1: PING-PONG (image <-> texte)
    pp_texts = []
    pp_images = [(image_id, 1.0)]
    current_vision = vision_feat
    
    for step in range(n_passes):
        # Vision -> Text
        text_e5_hat = workspace.translate('vision', 'text', current_vision)
        pp_texts.append(decode_e5_to_caption(text_e5_hat))
        
        # Text -> Vision
        current_vision = workspace.translate('text', 'vision', text_e5_hat)
        nn_id, nn_sim = find_nearest_image(current_vision)
        pp_images.append((nn_id, nn_sim))
    
    # MODE 2: ROUND-TRIP VISION (vision -> workspace -> vision)
    rt_captions = []
    rt_images = [(image_id, 1.0)]
    current_vision = vision_feat
    
    for step in range(n_passes):
        z_v = mod_v.encode(current_vision)
        w = mod_v.to_workspace(z_v)
        z_v_back = mod_v.from_workspace(w)
        current_vision = mod_v.decode(z_v_back)
        
        nn_id, nn_sim = find_nearest_image(current_vision)
        rt_images.append((nn_id, nn_sim))
        
        text_e5_hat = workspace.translate('vision', 'text', current_vision)
        rt_captions.append(decode_e5_to_caption(text_e5_hat))
    
    # AFFICHAGE MODE 1: PING-PONG
    n_cols = n_passes + 1
    fig, axes = plt.subplots(2, n_cols, figsize=(3.5 * n_cols, 8),
                             gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle(f"Ping-pong (img -> txt -> img -> ...)\nGT: {gt_caption[:80]}",
                 fontsize=11, fontweight='bold', y=1.02)
    
    for i in range(n_cols):
        img_id, sim = pp_images[i]
        img = load_coco_image(img_id)
        axes[0, i].imshow(img)
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9, fontweight='bold', color='green')
        else:
            axes[0, i].set_title(f"Pass {i} (sim={sim:.3f})", fontsize=9)
        axes[0, i].axis('off')
        
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].text(0.5, 0.5, f"GT:\n{gt_caption[:50]}",
                           transform=axes[1, i].transAxes, fontsize=8,
                           ha='center', va='center', style='italic', color='green')
        else:
            axes[1, i].text(0.5, 0.5, f"NN:\n{pp_texts[i-1][:50]}",
                           transform=axes[1, i].transAxes, fontsize=8,
                           ha='center', va='center')
    
    plt.tight_layout()
    plt.show()
    
    # AFFICHAGE MODE 2: ROUND-TRIP VISION
    fig, axes = plt.subplots(2, n_cols, figsize=(3.5 * n_cols, 8),
                             gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle("Round-trip vision (img -> w -> img -> w -> ...)",
                 fontsize=11, fontweight='bold', y=1.02)
    
    for i in range(n_cols):
        img_id, sim = rt_images[i]
        img = load_coco_image(img_id)
        axes[0, i].imshow(img)
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9, fontweight='bold', color='green')
        else:
            axes[0, i].set_title(f"Pass {i} (sim={sim:.3f})", fontsize=9)
        axes[0, i].axis('off')
        
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].text(0.5, 0.5, f"GT:\n{gt_caption[:50]}",
                           transform=axes[1, i].transAxes, fontsize=8,
                           ha='center', va='center', style='italic', color='green')
        else:
            axes[1, i].text(0.5, 0.5, f"Traduit:\n{rt_captions[i-1][:50]}",
                           transform=axes[1, i].transAxes, fontsize=8,
                           ha='center', va='center')
    
    plt.tight_layout()
    plt.show()

# Lancer sur 3 exemples
for idx in np.linspace(0, len(val_dataset)-1, 3, dtype=int):
    telephone_arabe(idx, n_passes=5)
    print("\n" + "=" * 80 + "\n")

## 5. Visualisation du Workspace (t-SNE)

On projette les representations workspace dans un espace 2D pour visualiser l'alignement.

In [ ]:
from sklearn.manifold import TSNE

# Prendre un sous-ensemble (t-SNE est lent sur >5k points)
N_SUBSET = 2000
indices = np.random.choice(len(all_w_vision), N_SUBSET, replace=False)

w_v_sub = all_w_vision[indices].numpy()
w_t_sub = all_w_text[indices].numpy()

# Concatener vision et texte
combined = np.concatenate([w_v_sub, w_t_sub], axis=0)

print(f"Running t-SNE on {len(combined)} points...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
embedded = tsne.fit_transform(combined)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

v_emb = embedded[:N_SUBSET]
t_emb = embedded[N_SUBSET:]

axes[0].scatter(v_emb[:, 0], v_emb[:, 1], c='tab:blue', alpha=0.3, s=5, label='Vision')
axes[0].scatter(t_emb[:, 0], t_emb[:, 1], c='tab:orange', alpha=0.3, s=5, label='Text')
axes[0].legend(fontsize=12)
axes[0].set_title('Workspace t-SNE: Vision vs Text')

# Lignes entre paires alignees
N_LINES = 100
line_indices = np.random.choice(N_SUBSET, N_LINES, replace=False)

axes[1].scatter(v_emb[:, 0], v_emb[:, 1], c='tab:blue', alpha=0.15, s=5)
axes[1].scatter(t_emb[:, 0], t_emb[:, 1], c='tab:orange', alpha=0.15, s=5)

for li in line_indices:
    axes[1].plot([v_emb[li, 0], t_emb[li, 0]],
                [v_emb[li, 1], t_emb[li, 1]],
                'k-', alpha=0.15, linewidth=0.5)

axes[1].set_title(f'Paires alignees ({N_LINES} exemples)\nLignes courtes = bon alignement')

plt.tight_layout()
plt.show()

## 6. Distribution des similarites

Histogramme des similarites cosine: paires positives (matchees) vs negatives.

In [ ]:
# Similarites positives (diagonale) vs negatives
pos_sims = sim_matrix.diagonal().numpy()

# Echantillonner des negatives
N = len(sim_matrix)
neg_indices = []
for _ in range(50000):
    i, j = np.random.randint(N), np.random.randint(N)
    if i != j:
        neg_indices.append((i, j))
neg_sample = np.array([sim_matrix[i, j].item() for i, j in neg_indices])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(neg_sample, bins=100, alpha=0.6, label=f'Negatives (mean={neg_sample.mean():.3f})', density=True, color='tab:red')
axes[0].hist(pos_sims, bins=100, alpha=0.6, label=f'Positives (mean={pos_sims.mean():.3f})', density=True, color='tab:green')
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution des similarites workspace')
axes[0].legend()

axes[1].boxplot([neg_sample, pos_sims], labels=['Negatives', 'Positives'])
axes[1].set_ylabel('Cosine Similarity')
axes[1].set_title('Separation positives vs negatives')

plt.tight_layout()
plt.show()

print(f"Positives: mean={pos_sims.mean():.4f}, std={pos_sims.std():.4f}")
print(f"Negatives: mean={neg_sample.mean():.4f}, std={neg_sample.std():.4f}")
print(f"Separation (difference des moyennes): {pos_sims.mean() - neg_sample.mean():.4f}")

## 7. Heatmap de similarite (sous-ensemble)

Visualisation directe de la matrice de similarite. La diagonale brillante = bon alignement.

In [ ]:
N_SHOW = 100
sub_sim = sim_matrix[:N_SHOW, :N_SHOW].numpy()

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
im = ax.imshow(sub_sim, cmap='viridis', aspect='auto')
ax.set_xlabel('Text samples')
ax.set_ylabel('Vision samples')
ax.set_title(f'Matrice de similarite workspace (premiers {N_SHOW} samples)\nDiagonale brillante = bon alignement')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 8. Diagnostics: normes et distributions des latents

Verification que les representations ne degenerent pas (collapse, normes asymetriques, etc.).

In [ ]:
@torch.no_grad()
def get_all_latents(dataloader):
    """Recupere latents et workspace reprs pour tout le val set."""
    z_v, z_t, w_v, w_t = [], [], [], []
    
    for batch in dataloader:
        vis = batch['vision'].to(device)
        txt = batch['text'].to(device)
        
        zv = workspace.domain_modules['vision'].encode(vis)
        zt = workspace.domain_modules['text'].encode(txt)
        wv = workspace.domain_modules['vision'].to_workspace(zv)
        wt = workspace.domain_modules['text'].to_workspace(zt)
        
        z_v.append(zv.cpu()); z_t.append(zt.cpu())
        w_v.append(wv.cpu()); w_t.append(wt.cpu())
    
    return (torch.cat(z_v), torch.cat(z_t),
            torch.cat(w_v), torch.cat(w_t))

z_vision, z_text, w_vision_raw, w_text_raw = get_all_latents(val_loader)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Normes des latents
axes[0, 0].hist(z_vision.norm(dim=-1).numpy(), bins=50, alpha=0.6, label='Vision z')
axes[0, 0].hist(z_text.norm(dim=-1).numpy(), bins=50, alpha=0.6, label='Text z')
axes[0, 0].set_title('Normes des latents (avant projection)')
axes[0, 0].legend()

# Normes workspace (avant normalisation)
axes[0, 1].hist(w_vision_raw.norm(dim=-1).numpy(), bins=50, alpha=0.6, label='Vision w')
axes[0, 1].hist(w_text_raw.norm(dim=-1).numpy(), bins=50, alpha=0.6, label='Text w')
axes[0, 1].set_title('Normes workspace (avant normalisation)')
axes[0, 1].legend()

# Variance par dimension (workspace)
axes[1, 0].bar(range(w_vision_raw.shape[1]), w_vision_raw.var(dim=0).numpy(), alpha=0.6, label='Vision', width=1)
axes[1, 0].bar(range(w_text_raw.shape[1]), w_text_raw.var(dim=0).numpy(), alpha=0.4, label='Text', width=1)
axes[1, 0].set_title('Variance par dimension du workspace')
axes[1, 0].set_xlabel('Dimension')
axes[1, 0].legend()

# Participation ratio
def participation_ratio(x):
    cov = torch.cov(x.T)
    eigenvalues = torch.linalg.eigvalsh(cov)
    eigenvalues = eigenvalues.clamp(min=0)
    pr = (eigenvalues.sum() ** 2) / (eigenvalues ** 2).sum()
    return pr.item()

pr_v = participation_ratio(w_vision_raw)
pr_t = participation_ratio(w_text_raw)
total_dims = w_vision_raw.shape[1]

axes[1, 1].bar(['Vision', 'Text'], [pr_v, pr_t], color=['tab:blue', 'tab:orange'])
axes[1, 1].axhline(y=total_dims, color='r', linestyle='--', label=f'Max ({total_dims})')
axes[1, 1].set_title(f'Dimensionnalite effective (participation ratio)\nMax = {total_dims}')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f"Participation ratio: Vision={pr_v:.1f}/{total_dims}, Text={pr_t:.1f}/{total_dims}")
print(f"  -> Vision utilise {pr_v/total_dims*100:.0f}% des dimensions")
print(f"  -> Text utilise {pr_t/total_dims*100:.0f}% des dimensions")

## 9. Resume

Recap de toutes les metriques en un coup d'oeil.

In [ ]:
print("=" * 50)
print("RESUME DES RESULTATS")
print("=" * 50)

print(f"\n--- Retrieval Vision -> Text ({N_EVAL} samples) ---")
for k, v in v2t.items():
    random_baseline = min(int(k.split('@')[1]), N_EVAL) / N_EVAL * 100
    print(f"  {k}: {v*100:.1f}%  (random: {random_baseline:.2f}%)")

print(f"\n--- Retrieval Text -> Vision ({N_EVAL} samples) ---")
for k, v in t2v.items():
    print(f"  {k}: {v*100:.1f}%")

print(f"\n--- Workspace Stats ---")
print(f"  Similarity positives (mean): {pos_sims.mean():.4f}")
print(f"  Similarity negatives (mean): {neg_sample.mean():.4f}")
print(f"  Separation: {pos_sims.mean() - neg_sample.mean():.4f}")
print(f"  Participation ratio vision: {pr_v:.1f}/{total_dims}")
print(f"  Participation ratio text: {pr_t:.1f}/{total_dims}")

print(f"\n--- Training Losses (finales) ---")
if 'eval_results' in ckpt:
    for k, v in ckpt['eval_results'].items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")